# 3. Tool Calling / Function Calling

Modern chat models can decide to call an external function instead of answering
directly. The model is given a list of available tools — their names, arguments,
and docstrings — and when it wants to use one, it returns a structured tool call
(name + arguments) instead of plain text. The calling code executes that function
and feeds the result back to the model for a final natural-language answer.

This notebook builds the bind-execute-respond loop **inline**, cell by cell —
reusing this project's `tools/*.py` functions (their docstrings are exactly what
a tool-calling model reads to decide which one fits), but not the app's
`langchain_demo.tool_calling`/`tool_utils` wrapper functions, so every step stays visible.

**Prerequisites:** Ollama running locally with `llama3.2` pulled. (The `web_search`
tool additionally needs `GOOGLE_API_KEY`/`GOOGLE_CSE_ID` in `.env` to actually
succeed — see `tools/search_tools.py` — but it degrades to an error string fed
back to the model otherwise, so this notebook works without it too.)

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Build the tool list and a tiny call-by-name helper

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

from models.chat_models.ollama_models import SupportedModel, get_chat_model
from tools.math_tools import adder, divider, multiplier, subtractor
from tools.string_tools import is_palindrome, reverse_text, word_count
from tools.search_tools import web_search

TOOLS = [adder, subtractor, multiplier, divider, reverse_text, word_count, is_palindrome, web_search]
TOOLS_BY_NAME = {tool.__name__: tool for tool in TOOLS}


def call_tool(name: str, args: dict):
    try:
        return TOOLS_BY_NAME[name](**args)
    except Exception as exc:
        return f"Error calling tool '{name}': {exc}"


llm = get_chat_model(SupportedModel.llama3_2)

## Round 1 — bind the tools and ask a question

In [ ]:
messages = [HumanMessage(content="what is 12 times 7?")]
ai_message = llm.bind_tools(TOOLS).invoke(messages)
messages.append(ai_message)

print("content:", repr(ai_message.content))
print("tool_calls:", ai_message.tool_calls)

## Round 2 — execute the requested tool call(s), feed results back, ask for the final answer

In [ ]:
for tool_call in ai_message.tool_calls:
    result = call_tool(tool_call["name"], tool_call["args"])
    print(f"{tool_call['name']}({tool_call['args']}) = {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

if ai_message.tool_calls:
    final_message = llm.invoke(messages)
    print()
    print("final answer:", final_message.content)
else:
    print("final answer:", ai_message.content)

## 🧪 Playground

**1. A string tool** — try `"reverse the word hello"` from Round 1 onward (fresh `messages` list).

In [ ]:
# TODO: messages = [HumanMessage(content="reverse the word hello")] ... repeat rounds 1-2


**2. A two-step math request** — try `"add 15 and 27, then multiply the result by 2"`. Does the model chain two tool calls, or does it get confused? (This is a documented small-model limitation — see `docs/langchain/03-tool-calling.md`'s Gotchas.)

In [ ]:
# TODO: try the two-step request and inspect ai_message.tool_calls


**3. No tool needed** — try `"what's your favorite color?"` and confirm `tool_calls` comes back empty.

In [ ]:
# TODO: try a question that needs no tool at all
